In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("loan_prediction_dataset.csv")
df

print("Shape:", df.shape)
df.head()

In [ ]:
# 🔍 Basic information
print("="*50)
print(" DATASET INFORMATION")
print("="*50)
print("\nColumn Names:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)

print("\n" + "="*50)
print("MISSING VALUES CHECK")
print("="*50)
print(df.isnull().sum())
print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")

print("\n" + "="*50)
print(" BASIC STATISTICS")
print("="*50)
df.describe()

In [ ]:
# Numerical columns ke liye median
numerical_cols = ['Age', 'Income', 'Credit_Score', 'Loan_Amount', 'Loan_Term']

for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
        print(f"✅ {col} - Missing values filled with median")

# Categorical column (Employment_Status) ke liye mode
if df['Employment_Status'].isnull().sum() > 0:
    df['Employment_Status'].fillna(df['Employment_Status'].mode()[0], inplace=True)
    print("✅ Employment_Status - Missing values filled with mode")

# Final check
print("\n" + "="*50)
print("REMAINING MISSING VALUES:", df.isnull().sum().sum())
print("="*50)

In [ ]:
# Style set
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

# Loan Approval Distribution
print("\n Loan Approval Distribution:")
print(df['Loan_Approved'].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x='Loan_Approved', data=df, palette='Set1')
plt.title('Loan Approval Distribution (0=Rejected, 1=Approved)')
plt.xlabel('Loan Approved')
plt.ylabel('Count')
plt.xticks([0, 1], ['Rejected', 'Approved'])
plt.show()

# Loan Amount Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Loan_Amount'], bins=30, kde=True, color='teal')
plt.title('Distribution of Loan Amount')
plt.xlabel('Loan Amount')
plt.ylabel('Frequency')
plt.show()

# Income vs Loan Approval
plt.figure(figsize=(8, 5))
sns.boxplot(x='Loan_Approved', y='Income', data=df, palette='Set2')
plt.title('Income vs Loan Approval Status')
plt.xlabel('Loan Approved (0=No, 1=Yes)')
plt.ylabel('Income')
plt.xticks([0, 1], ['Rejected', 'Approved'])
plt.show()

# Credit Score vs Loan Approval
plt.figure(figsize=(8, 5))
sns.boxplot(x='Loan_Approved', y='Credit_Score', data=df, palette='viridis')
plt.title('Credit Score vs Loan Approval Status')
plt.xlabel('Loan Approved (0=No, 1=Yes)')
plt.ylabel('Credit Score')
plt.xticks([0, 1], ['Rejected', 'Approved'])
plt.show()

# Employment Status vs Loan Approval
plt.figure(figsize=(8, 5))
sns.countplot(x='Employment_Status', hue='Loan_Approved', data=df, palette='coolwarm')
plt.title('Loan Approval by Employment Status')
plt.xlabel('Employment Status')
plt.ylabel('Count')
plt.legend(['Rejected', 'Approved'])
plt.show()

# Age Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Age'], bins=20, kde=True, color='orange')
plt.title('Age Distribution of Applicants')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Features (X) aur Target (y) alag kardye
X = df.drop('Loan_Approved', axis=1)
y = df['Loan_Approved']

print("Features (X) columns:", X.columns.tolist())
print("\nTarget (y) unique values:", y.unique())

# Categorical variable (Employment_Status) ko encode kardia
# Label Encoding use karenge
label_encoder = LabelEncoder()
X['Employment_Status'] = label_encoder.fit_transform(X['Employment_Status'])

print("\n✅ Employment_Status encoded:")
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# 🔍 Check karein ki sab numerical ho gaye
print("\n Data Types After Encoding:")
print(X.dtypes)

# 📏 Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% test data
    random_state=42,      # Reproducibility ke liye
    stratify=y            # Class balance maintain karne ke liye
)

print("\n" + "="*50)
print("TRAIN-TEST SPLIT")
print("="*50)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")

# 📐 Feature Scaling (Standardization)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("\n✅ Features scaled successfully!")
print("Note: Scaling se har feature ka mean=0 aur std=1 ho jata hai")

In [ ]:
#  Logistic Regression Model
print("="*50)
print(" TRAINING LOGISTIC REGRESSION MODEL")
print("="*50)

model = LogisticRegression(
    random_state=42,
    class_weight='balanced',  # Agar classes imbalanced hain toh helpful hai
    max_iter=1000
)

# 🏋️ Model train
model.fit(X_train, y_train)

print("✅ Model trained successfully!")

#  Model coefficients (feature importance)
print("\n" + "="*50)
print("FEATURE IMPORTANCE (Coefficients)")
print("="*50)
feature_names = X.columns.tolist()
coefficients = model.coef_[0]

for feature, coef in zip(feature_names, coefficients):
    print(f"{feature:20} : {coef:.4f}")

In [ ]:
#  Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability of approval

print("="*50)
print(" MODEL EVALUATION")
print("="*50)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n ACCURACY: {accuracy*100:.2f}%")
print(f"Correct predictions: {accuracy_score(y_test, y_pred, normalize=False)} out of {len(y_test)}")

# Confusion Matrix
print("\n" + "="*50)
print(" CONFUSION MATRIX")
print("="*50)
cm = confusion_matrix(y_test, y_test)
cm = confusion_matrix(y_test, y_pred)

print(f"\nTrue Negatives (Correctly Rejected):  {cm[0,0]}")
print(f"False Positives (Wrongly Approved):    {cm[0,1]}")
print(f"False Negatives (Wrongly Rejected):    {cm[1,0]}")
print(f"True Positives (Correctly Approved):   {cm[1,1]}")

# Confusion Matrix Visualization
plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Rejected', 'Approved'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Logistic Regression')
plt.grid(False)
plt.show()

# Classification Report
print("\n" + "="*50)
print(" CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))

# Additional Metrics
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n" + "="*50)
print(" ADDITIONAL METRICS")
print("="*50)
print(f"Precision: {precision:.4f} (Approved loans mein se kitne sahi the)")
print(f"Recall:    {recall:.4f} (Total approved ko kitna capture kiya)")
print(f"F1 Score:  {f1:.4f} (Precision aur Recall ka harmonic mean)")

In [ ]:
# Sample Predictions
print("="*50)
print(" SAMPLE PREDICTIONS")
print("="*50)

# Test set se first 10 samples
# y_test abhi bhi pandas Series hai, isliye uska index use kra
sample_indices = y_test[:10].index

print(f"\nShowing predictions for {len(sample_indices)} samples:\n")

for i, idx in enumerate(sample_indices):
    actual = y_test.loc[idx]
    predicted = y_pred[i]  # y_pred numpy array hai, isliye index se access
    proba = y_pred_proba[i]

    status = "✅ CORRECT" if actual == predicted else "❌ WRONG"

    print(f"Sample {i+1}:")
    print(f"  Actual: {actual} ({'Approved' if actual==1 else 'Rejected'})")
    print(f"  Predicted: {predicted} ({'Approved' if predicted==1 else 'Rejected'})")
    print(f"  Approval Probability: {proba*100:.2f}%")
    print(f"  Status: {status}")
    print("-" * 40)

In [ ]:
# Decision Tree Model
print("="*50)
print("🌲 TRAINING DECISION TREE MODEL")
print("="*50)

dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5,            # Overfitting rokne ke liye
    min_samples_split=10,
    min_samples_leaf=5
)

dt_model.fit(X_train, y_train)

# Predictions
dt_pred = dt_model.predict(X_test)

# Evaluation
dt_accuracy = accuracy_score(y_test, dt_pred)
print(f"\n Decision Tree Accuracy: {dt_accuracy*100:.2f}%")
print(f" Logistic Regression Accuracy: {accuracy*100:.2f}%")

# Comparison
print("\n" + "="*50)
print(" MODEL COMPARISON")
print("="*50)
print(f"{'Model':<25} {'Accuracy':<15}")
print("-"*40)
print(f"{'Logistic Regression':<25} {accuracy*100:.2f}%")
print(f"{'Decision Tree':<25} {dt_accuracy*100:.2f}%")

# Confusion Matrix for Decision Tree
plt.figure(figsize=(8, 6))
cm_dt = confusion_matrix(y_test, dt_pred)
disp_dt = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=['Rejected', 'Approved'])
disp_dt.plot(cmap='Greens', values_format='d')
plt.title('Confusion Matrix - Decision Tree')
plt.grid(False)
plt.show()